# Stage 18 P-as-Wake Sensitivity Analysis

The primary DREAMT workflow excludes `P` preparation epochs because they are not necessarily PSG-scored Wake. This notebook reports a validation-only sensitivity analysis that follows the original DREAMT paper convention by mapping `P` to `Wake`. The analysis uses one representative model: the Stage 14 square-root-weighted multiscale residual feature-fusion CNN.

In [ ]:
# ruff: noqa
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.p_as_wake_sensitivity import (
    DEFAULT_STAGE18_OUTPUT_DIR,
    load_primary_stage14_weighted_summary,
    load_stage18_p_as_wake_summary,
    run_stage18_p_as_wake_sensitivity,
)

stage18_output_dir = repo_root / DEFAULT_STAGE18_OUTPUT_DIR
stage18_figures_dir = stage18_output_dir / "figures"
stage18_figures_dir.mkdir(parents=True, exist_ok=True)

raw_dir = repo_root / "data" / "raw"
split_assignments_path = repo_root / "data" / "interim" / "split_assignments.csv"
raw_artifacts_available = raw_dir.exists() and split_assignments_path.exists()
raw_artifacts_available

## Guarded Stage 18 Run

In [ ]:
RUN_STAGE18_P_AS_WAKE_SENSITIVITY = False
SKIP_COMPLETED_STAGE18 = True
OVERWRITE_STAGE18_EPOCH_INDEX = False
OVERWRITE_STAGE18_FEATURE_TABLES = False

if RUN_STAGE18_P_AS_WAKE_SENSITIVITY and raw_artifacts_available:
    stage18_summary = run_stage18_p_as_wake_sensitivity(
        raw_dir=raw_dir,
        split_assignments_path=split_assignments_path,
        output_dir=stage18_output_dir,
        overwrite_epoch_index=OVERWRITE_STAGE18_EPOCH_INDEX,
        overwrite_feature_tables=OVERWRITE_STAGE18_FEATURE_TABLES,
        skip_completed=SKIP_COMPLETED_STAGE18,
    )
    display(stage18_summary)
elif raw_artifacts_available:
    print("Stage 18 is configured but not run in this notebook execution.")
    print("output directory:", stage18_output_dir)
else:
    print("Local raw DREAMT artifacts are unavailable; Stage 18 run is skipped.")

## Epoch and Label Counts

In [ ]:
stage18_epoch_index_path = stage18_output_dir / "epoch_index_p_as_wake_train_validation.csv"

if stage18_epoch_index_path.exists():
    stage18_epoch_index = pd.read_csv(stage18_epoch_index_path, dtype={"participant_id": str})
    valid_stage18_epochs = stage18_epoch_index[stage18_epoch_index["is_valid_epoch"].astype(bool)].copy()
    stage18_label_counts = (
        valid_stage18_epochs.groupby(["split", "mapped_label"])
        .size()
        .unstack(fill_value=0)
        .reindex(index=["train", "validation"])
    )
    display(stage18_label_counts)
else:
    stage18_label_counts = pd.DataFrame()
    print("Stage 18 epoch index not found yet.")

## Validation Metric Comparison

In [ ]:
primary_stage14_summary = load_primary_stage14_weighted_summary(
    repo_root / "results" / "stage14_multiscale_fusion_cnn_sqrt_weighted"
)
stage18_summary = load_stage18_p_as_wake_summary(stage18_output_dir)

metric_columns = ["macro_f1", "balanced_accuracy", "Wake_f1", "Non_REM_f1", "REM_f1"]
available_metrics = [
    metric
    for metric in metric_columns
    if metric in primary_stage14_summary.columns and metric in stage18_summary.columns
]

if not primary_stage14_summary.empty and not stage18_summary.empty and available_metrics:
    primary_row = primary_stage14_summary.iloc[0]
    stage18_row = stage18_summary.iloc[0]
    comparison = pd.DataFrame(
        [
            {"analysis": "Primary: drop P", **{metric: primary_row[metric] for metric in available_metrics}},
            {"analysis": "Sensitivity: P as Wake", **{metric: stage18_row[metric] for metric in available_metrics}},
        ]
    )
    delta = comparison.set_index("analysis").diff().iloc[-1].rename("sensitivity_minus_primary")
    display(comparison)
    display(delta.to_frame())
else:
    comparison = pd.DataFrame()
    delta = pd.Series(dtype=float)
    print("Primary and Stage 18 validation summaries are both needed for comparison.")

In [ ]:
if not comparison.empty:
    ax = comparison.set_index("analysis")[available_metrics].T.plot(kind="bar", figsize=(9, 4))
    ax.set_ylabel("Validation score")
    ax.set_xlabel("")
    ax.set_ylim(0, 1)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.savefig(stage18_figures_dir / "stage18_validation_metric_comparison.png", dpi=200)
    plt.show()

if not delta.empty:
    ax = delta.plot(kind="bar", figsize=(8, 3), color="#4C78A8")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylabel("Sensitivity minus primary")
    ax.set_xlabel("")
    plt.tight_layout()
    plt.savefig(stage18_figures_dir / "stage18_validation_metric_deltas.png", dpi=200)
    plt.show()